# Fase 2: Preprocesamiento y Transformación de Logs de Seguridad
**Proyecto:** Análisis de datos de amenazas de ciberseguridad  
**Integrantes:** Jennifer Nilo – Patricio Núñez  

## 1. Contexto y Objetivos
En la operación de un Centro de Operaciones de Seguridad (SOC), la calidad de los datos ingeridos es fundamental. Los logs con valores nulos o formatos incorrectos generan falsos positivos y aumentan la fatiga de alertas. 

El objetivo de esta Fase 2 es aplicar un pipeline secuencial para depurar nuestro dataset de 6 millones de registros mediante:
1. **Exploración e Ingesta:** Carga segura de datos.
2. **Limpieza:** Tratamiento de valores nulos (NA).
3. **Transformación:** Conversión de tipos de datos (*Casting*).
4. **Normalización:** Escalamiento de variables numéricas.
5. **Validación:** Comprobación de la integridad del dataset final.

In [1]:
# Importación de librerías base
import pandas as pd
import numpy as np

# Fijar la semilla de aleatoriedad para garantizar la reproducibilidad del experimento
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Librerías importadas. Entorno de la Fase 2 inicializado.")

Librerías importadas. Entorno de la Fase 2 inicializado.


## 2. Obtención e Ingesta de Datos
Para procesar de manera eficiente los 6 millones de registros sin saturar la memoria RAM del entorno, estructuramos la ingesta mediante una carga optimizada por bloques (*chunks*). Utilizando el parámetro `chunksize` de Pandas, el archivo masivo se lee en fracciones manejables de 500.000 filas a la vez. Además, mantenemos el parámetro `low_memory=False` dentro de la lectura de cada bloque para garantizar que los tipos de datos se detecten con precisión y sin advertencias, concatenando finalmente todas las partes en un único conjunto de datos consolidado.

In [2]:
def cargar_datos_en_chunks(ruta, tamano_chunk=500000):
    """
    Función para cargar el CSV masivo en bloques (chunks) estructurados, optimizando el consumo de memoria RAM durante la ingesta.
    """
    print(f"Iniciando carga optimizada por chunks desde: {ruta}...")
    
    lista_chunks = []
    
    # Iteramos sobre el archivo leyendo en bloques definidos por tamano_chunk
    for i, chunk in enumerate(pd.read_csv(ruta, chunksize=tamano_chunk, low_memory=False)):
        lista_chunks.append(chunk)
        print(f"  -> Bloque {i+1} cargado en memoria: {len(chunk):,} filas.")
        
    # Unimos todos los bloques en un solo DataFrame consolidado
    print("Concatenando bloques...")
    dataframe_final = pd.concat(lista_chunks, ignore_index=True)
    
    print(f"¡Carga exitosa! Filas totales: {dataframe_final.shape[0]:,}, Columnas: {dataframe_final.shape[1]}")
    return dataframe_final

# La funcion se ejecuta con los datos almacenados en /data/raw
ruta_archivo = '../data/raw/cybersecurity_threat_detection_logs.csv'
# Procesaremos de a medio millón de filas a la vez
df_crudo = cargar_datos_en_chunks(ruta_archivo, tamano_chunk=500000)

# Visualización del dataset original para contrastar con el resultado final
print("\nMuestra de los datos crudos originales (Antes del preprocesamiento):")
display(df_crudo.head())

Iniciando carga optimizada por chunks desde: ../data/raw/cybersecurity_threat_detection_logs.csv...
  -> Bloque 1 cargado en memoria: 500,000 filas.
  -> Bloque 2 cargado en memoria: 500,000 filas.
  -> Bloque 3 cargado en memoria: 500,000 filas.
  -> Bloque 4 cargado en memoria: 500,000 filas.
  -> Bloque 5 cargado en memoria: 500,000 filas.
  -> Bloque 6 cargado en memoria: 500,000 filas.
  -> Bloque 7 cargado en memoria: 500,000 filas.
  -> Bloque 8 cargado en memoria: 500,000 filas.
  -> Bloque 9 cargado en memoria: 500,000 filas.
  -> Bloque 10 cargado en memoria: 500,000 filas.
  -> Bloque 11 cargado en memoria: 500,000 filas.
  -> Bloque 12 cargado en memoria: 500,000 filas.
Concatenando bloques...
¡Carga exitosa! Filas totales: 6,000,000, Columnas: 10


## 3. Limpieza Exhaustiva (Manejo de Valores NA)
La depuración de valores nulos se divide en dos criterios operativos:
* **Descarte de registros inválidos:** Si un log no contiene dirección IP o Timestamp, pierde su valor analítico y es eliminado.
* **Imputación de metadatos:** Si faltan datos complementarios (como el `user_agent`), se rellenan con un valor por defecto ("Desconocido") para no perder el resto de la información útil de esa fila.

In [3]:
def limpiar_nulos(df):
    """Elimina filas sin IP o Timestamp, y rellena los vacíos en otras columnas."""
    df_limpio = df.copy()
    
    # 1. Columnas críticas (Si faltan, el log es ruido)
    columnas_obligatorias = ['timestamp', 'source_ip', 'dest_ip']
    df_limpio = df_limpio.dropna(subset=columnas_obligatorias)
    
    # 2. Relleno de datos faltantes en columnas secundarias
    if 'user_agent' in df_limpio.columns:
        df_limpio['user_agent'] = df_limpio['user_agent'].fillna('Desconocido')
        
    if 'request_path' in df_limpio.columns:
        df_limpio['request_path'] = df_limpio['request_path'].fillna('/')
        
    return df_limpio

# Ejecucion de limpieza
df_sin_nulos = limpiar_nulos(df_crudo)
print(f"Filas retenidas después de limpiar nulos: {df_sin_nulos.shape[0]:,}")

Filas retenidas después de limpiar nulos: 6,000,000


## 3.5 Validación de Integridad de Direcciones IP
Los sensores de red ocasionalmente pueden generar logs corruptos donde los campos de origen y destino no contienen direcciones IP reales, sino fragmentos de texto o errores de codificación. Para evitar que esta "basura" contamine el *Feature Engineering* posterior, aplicamos un filtro de expresiones regulares (Regex) que descartará cualquier registro cuyas IPs no cumplan con la estructura estándar de una IPv4 (X.X.X.X).

In [4]:
def filtrar_ips_invalidas(df):
    """
    Filtra y descarta los registros donde source_ip o dest_ip no tengan un formato de dirección IPv4 válido (0-255 por octeto).
    """
    df_limpio = df.copy()
    
    # Expresión regular estricta para IPv4 real (0-255)
    regex_ipv4 = r'^(?:(?:25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)\.){3}(?:25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)$'
    
    # Verificamos qué filas cumplen con el formato exacto en ambas columnas
    mascara_source = df_limpio['source_ip'].astype(str).str.match(regex_ipv4)
    mascara_dest = df_limpio['dest_ip'].astype(str).str.match(regex_ipv4)
    
    # Filtramos manteniendo solo las filas correctas
    df_filtrado = df_limpio[mascara_source & mascara_dest]
    
    # Calcular el numero de registros borrados avalores no validos como direcciones IP IPv4
    borradas = len(df_limpio) - len(df_filtrado)
    if borradas > 0:
        print(f"Se descartaron {borradas:,} filas por contener IPs fuera de rango o corruptas.")
    else:
        print("Validación completada: Todas las IPs tienen un formato IPv4 estricto.")
        
    return df_filtrado

# Ejecutamos este filtro pasándole el dataset que ya no tiene nulos (Paso 3)
df_ips_validas = filtrar_ips_invalidas(df_sin_nulos)

print(f"Filas retenidas después de validar IPs: {df_ips_validas.shape[0]:,}")

Validación completada: Todas las IPs tienen un formato IPv4 estricto.
Filas retenidas después de validar IPs: 6,000,000


## 4. Transformación de Variables (Casting)
Los modelos de datos necesitan interpretar las fechas como tiempo real y no solo leerlas como cadenas de texto. Por esta razón, convertimos la información de la columna timestamp al formato de fechas propio de Pandas (datetime).

In [5]:
def transformar_fechas(df):
    """Convierte la columna de texto timestamp a un formato de tiempo."""
    df_transformado = df.copy()
    
    # errors='coerce' convierte cualquier texto corrupto en un valor nulo (NaT)
    df_transformado['timestamp'] = pd.to_datetime(df_transformado['timestamp'], errors='coerce')
    
    # Eliminar las filas que hayan quedado con fechas inválidas
    df_transformado = df_transformado.dropna(subset=['timestamp'])
    
    return df_transformado

# Ejecutar el casting
df_con_fechas = transformar_fechas(df_ips_validas)
print("Tipos de datos actualizados:")
print(df_con_fechas[['timestamp', 'source_ip']].dtypes)

Tipos de datos actualizados:
timestamp    datetime64[us]
source_ip               str
dtype: object


## 4.5 Tratamiento de Valores Atípicos (Outliers)
El volumen de tráfico de red (`bytes_transferred`) puede presentar picos extremos legítimos (exfiltración) o errores de registro. Para evitar que estos valores extremos compriman excesivamente nuestra normalización Min-Max, aplicaremos una técnica de limitación (Clipping) al percentil 99. Esto preserva la anomalía para el algoritmo sin distorsionar la escala global.

In [6]:
def tratar_atipicos(df):
    """Aplica clipping al percentil 99 en variables de volumen de tráfico."""
    df_out = df.copy()
    
    if 'bytes_transferred' in df_out.columns:
        # Calculamos el límite superior (percentil 99)
        limite_sup = df_out['bytes_transferred'].quantile(0.99)
        
        # Reemplazamos los valores que superen el límite por el valor del límite
        df_out['bytes_transferred'] = np.where(
            df_out['bytes_transferred'] > limite_sup, 
            limite_sup, 
            df_out['bytes_transferred']
        )
        
    return df_out

# Ejecutamos el tratamiento de atípicos
df_sin_atipicos = tratar_atipicos(df_con_fechas)
print("Tratamiento de valores atípicos completado.")

Tratamiento de valores atípicos completado.


## 5. Normalización de Datos
Para preparar el dataset para un futuro modelado algorítmico, aplicamos un escalamiento **Min-Max** a la variable `bytes_transferred`. Esto comprime todos los valores de transferencia de red en un rango entre 0 y 1, evitando que flujos inusualmente pesados sesguen los modelos de detección.

In [7]:
def escalar_bytes(df):
    """Escala los bytes transferidos a un valor entre 0 y 1 para facilitar el modelado."""
    df_escalado = df.copy()
    
    if 'bytes_transferred' in df_escalado.columns:
        minimo = df_escalado['bytes_transferred'].min()
        maximo = df_escalado['bytes_transferred'].max()
        
        # Fórmula matemática Min-Max
        if maximo > minimo:
            df_escalado['bytes_escalados'] = (df_escalado['bytes_transferred'] - minimo) / (maximo - minimo)
        else:
            df_escalado['bytes_escalados'] = 0.0
            
    return df_escalado

# Ejecutar la normalización
df_escalado = escalar_bytes(df_sin_atipicos)
print("Escalamiento completado. Muestra de los datos transformados:")
display(df_escalado[['timestamp', 'bytes_transferred', 'bytes_escalados']].head())

Escalamiento completado. Muestra de los datos transformados:


,timestamp,bytes_transferred,bytes_escalados
0,2024-05-01,10889.0,0.218396
1,2024-07-18,36522.0,0.737273
2,2024-04-07,20652.0,0.416024
3,2024-10-26,5350.0,0.106273
4,2024-10-31,40691.0,0.821664


## 6. Ingeniería de Características y Codificación (Feature Engineering)
Para que los modelos funcionen correctamente, es obligatorio que el archivo final sea una matriz estrictamente numérica. Aunque en ciberseguridad operativa las columnas de texto (como las direcciones IP o el navegador) son vitales para el análisis humano, las fórmulas matemáticas no pueden procesarlas. 

Para resolver esto sin perder el valor de la información, aplicaremos dos grandes pasos:

**1. Extracción de comportamientos (Feature Extraction):**
En lugar de dejar el texto crudo, lo "traduciremos" a indicadores tácticos numéricos:
* **Direcciones IP (`source_ip`, `dest_ip`):** Evaluaremos si la IP pertenece a una red privada (interna). Crearemos una columna nueva con un `1` si es interna, y un `0` si es pública.
* **Agente de usuario (`user_agent`):** Buscaremos patrones que delaten el uso de herramientas automatizadas de escaneo o ataque (como Nmap, curl o sqlmap), asignando un `1` si se detectan.
* **Ruta de petición (`request_path`):** Mediremos el largo (cantidad de caracteres) de la ruta, ya que los ciberataques suelen enviar peticiones inusualmente largas.
* *Limpieza:* Una vez extraídos estos datos numéricos, descartaremos las columnas de texto originales para liberar memoria.

**2. Transformación de categorías (Encoding):**
* **Variables sin jerarquía (`action`, `log_type`, `protocol`):** Aplicamos la técnica de *One-Hot Encoding*. Al agregar el protocolo a este paso, separamos sus valores (TCP, UDP, ICMP) en columnas nuevas de ceros y unos, evitando que el sistema asuma por error que un protocolo es "mayor" que otro.
* **Variable objetivo o nivel de amenaza (`threat_label`):** Aplicamos *Label Encoding*. Como el nivel de amenaza sí tiene un orden de riesgo, reemplazamos el texto directamente por números que reflejan su severidad (`benign` = 0, `suspicious` = 1, `malicious` = 2).

In [8]:
def codificar_y_extraer_caracteristicas(df):
    """
    Transforma texto en variables numéricas y extrae comportamientos de red. Garantiza que la salida sea una matriz 100% numérica.
    """
    df_feat = df.copy()
    
    # 1. Extracción de lógica de IPs (¿Es tráfico interno?)
    # Expresión regular simple para rangos privados (10.x, 172.16-31.x, 192.168.x)
    regex_privada = r'^(?:10\.|172\.(?:1[6-9]|2[0-9]|3[0-1])\.|192\.168\.)'
    df_feat['is_internal_source'] = df_feat['source_ip'].astype(str).str.contains(regex_privada).astype(int)
    df_feat['is_internal_dest'] = df_feat['dest_ip'].astype(str).str.contains(regex_privada).astype(int)
    
    # 2. Análisis del User Agent (¿Es una herramienta automatizada?)
    herramientas = 'nmap|curl|python|wget|postman|nikto|sqlmap'
    df_feat['is_automated_ua'] = df_feat['user_agent'].astype(str).str.lower().str.contains(herramientas).astype(int)
    
    # 3. Análisis del Request Path (Longitud y complejidad)
    df_feat['path_length'] = df_feat['request_path'].astype(str).str.len()
    
    # 4. Eliminamos las columnas de texto crudo originales que ya procesamos
    columnas_texto_a_borrar = ['source_ip', 'dest_ip', 'user_agent', 'request_path']
    df_feat = df_feat.drop(columns=columnas_texto_a_borrar, errors='ignore')
    
    # 5. One-Hot Encoding para categorías (¡Ahora incluimos protocol!)
    columnas_nominales = ['action', 'log_type', 'protocol']
    df_feat = pd.get_dummies(df_feat, columns=columnas_nominales, dtype=int)
    
    # 6. Label Encoding para la variable objetivo (Target)
    if 'threat_label' in df_feat.columns:
        mapa_amenazas = {'benign': 0, 'suspicious': 1, 'malicious': 2}
        df_feat['threat_label_encoded'] = df_feat['threat_label'].map(mapa_amenazas)
        df_feat = df_feat.drop(columns=['threat_label'])
        
    return df_feat

# Ejecutamos la transformación completa
df_final = codificar_y_extraer_caracteristicas(df_escalado)

print("Feature Engineering completado. Muestra de las columnas numéricas finales:")
display(df_final.head())

Feature Engineering completado. Muestra de las columnas numéricas finales:


,timestamp,bytes_transferred,bytes_escalados,is_internal_source,is_internal_dest,is_automated_ua,path_length,action_allowed,action_blocked,log_type_application,log_type_firewall,log_type_ids,protocol_FTP,protocol_HTTP,protocol_HTTPS,protocol_ICMP,protocol_SSH,protocol_TCP,protocol_UDP,threat_label_encoded
0,2024-05-01,10889.0,0.218396,1,1,1,1,0,1,0,1,0,0,0,0,0,0,1,0,0
1,2024-07-18,36522.0,0.737273,1,1,1,1,0,1,1,0,0,0,0,0,1,0,0,0,0
2,2024-04-07,20652.0,0.416024,1,1,0,6,1,0,1,0,0,0,1,0,0,0,0,0,0
3,2024-10-26,5350.0,0.106273,1,1,0,6,1,0,1,0,0,0,1,0,0,0,0,0,0
4,2024-10-31,40691.0,0.821664,1,1,0,1,1,0,1,0,0,0,0,0,1,0,0,0,0


## 7. Validación Técnica del Código
El último paso consiste en verificar que todas las transformaciones aplicadas a lo largo de este *notebook* resulten en un *dataset* consistente, sin nulos residuales y con los tipos de datos correctos, asegurando su trazabilidad para la Fase 3.

In [9]:
# Comprobación de resultados usando aserciones (assert)
print("--- INICIANDO VALIDACIÓN TÉCNICA ---")

try:
    # 1. Comprobación de Numericidad Total (select_dtypes)
    # Excluimos 'timestamp' porque es fecha, el resto DEBE ser número
    columnas_no_numericas = df_final.select_dtypes(include=['object', 'string']).columns
    assert len(columnas_no_numericas) == 0, f"Fallo de numericidad. Quedan columnas de texto: {list(columnas_no_numericas)}"
    print("[OK] El dataset es 100% numérico (matriz matemática validada).")
    
    # 2. Comprobación de Nulos
    nulos_restantes = df_final.isnull().sum().sum()
    assert nulos_restantes == 0, f"Fallo de integridad. Quedan {nulos_restantes} valores nulos."
    print("[OK] Ausencia total de valores nulos validada.")
    
    # 3. Comprobación de Escalamiento
    min_esc = df_final['bytes_escalados'].min()
    max_esc = df_final['bytes_escalados'].max()
    assert min_esc >= 0.0 and max_esc <= 1.0, f"Fallo de normalización. Rango actual: {min_esc} a {max_esc}"
    print(f"[OK] Rango de escalamiento Min-Max correcto ({min_esc:.1f} a {max_esc:.1f}).")
    
    print("\n VALIDACIÓN EXITOSA: El dataset cumple con todos los criterios de la rúbrica y está listo para ML.")

except AssertionError as error_validacion:
    print(f"\n ERROR CRÍTICO DE VALIDACIÓN: {error_validacion}")
except Exception as e:
    print(f"\n ERROR INESPERADO: {e}")

--- INICIANDO VALIDACIÓN TÉCNICA ---
[OK] El dataset es 100% numérico (matriz matemática validada).
[OK] Ausencia total de valores nulos validada.
[OK] Rango de escalamiento Min-Max correcto (0.0 a 1.0).

 VALIDACIÓN EXITOSA: El dataset cumple con todos los criterios de la rúbrica y está listo para ML.


## 8. Exportación del Dataset Procesado
Para garantizar la eficiencia de las fases posteriores, el dataset limpio, casteado y normalizado se exporta al directorio `data/processed/`. Esto evitará tener que repetir el costo computacional de esta fase durante la Fase 3.

In [10]:
# Definimos la ruta de salida
ruta_salida = '../data/processed/cybersecurity_logs_procesados.csv'

print("Filtrando columnas redundantes antes de exportar...")
# Se elimina la columna original porque ya tenemos la escalada
columnas_a_eliminar = ['bytes_transferred'] 

df_exportar = df_final.drop(columns=columnas_a_eliminar)

print("Iniciando exportación del dataset procesado...")
df_exportar.to_csv(ruta_salida, index=False)

print(f"¡Exportación exitosa! El dataset está listo en: {ruta_salida}")

Filtrando columnas redundantes antes de exportar...
Iniciando exportación del dataset procesado...
¡Exportación exitosa! El dataset está listo en: ../data/processed/cybersecurity_logs_procesados.csv
